In [31]:
#Set up environment for dependencies

In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Optional: to display all columns
pd.set_option('display.max_columns', None)

In [33]:
#Load the data
data = pd.read_csv("..\\data\\churn_prediction_data.csv")

## Basic Information

In [34]:
print("Dataset shape:", data.shape)
print("\nColumn names:")
print(data.columns.tolist())
print("\nFirst few rows:")
data.head()

Dataset shape: (7043, 21)

Column names:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

First few rows:


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Data Types and Missing Values

In [35]:
# Data types and missing values
print(data.info())
print("\nMissing Values:")
print(data.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


## Exploring Missing Values in TotalCharges

In [36]:
"""First look to see if there are empty strings or spaces that need to be dealt with before converting"""
print("Unique TotalCharges values (first 20):")
print(data["TotalCharges"].unique()[:20])

#check for any non-numeric values
print("\nAny non-numeric values?")
non_numeric = data[pd.to_numeric(data["TotalCharges"], errors="coerce").isna()]
print(f"Found {len(non_numeric)} non-numeric values")
if len(non_numeric) > 0:
    print(non_numeric["TotalCharges"].unique)

Unique TotalCharges values (first 20):
['29.85' '1889.5' '108.15' '1840.75' '151.65' '820.5' '1949.4' '301.9'
 '3046.05' '3487.95' '587.45' '326.8' '5681.1' '5036.3' '2686.05'
 '7895.15' '1022.95' '7382.25' '528.35' '1862.9']

Any non-numeric values?
Found 11 non-numeric values
<bound method Series.unique of 488      
753      
936      
1082     
1340     
3331     
3826     
4380     
5218     
6670     
6754     
Name: TotalCharges, dtype: object>


In [37]:
#Get the TotalCharges values for those problematic rows
problematic_indices = pd.to_numeric(data["TotalCharges"], errors="coerce").isna()
print("Non-numeric TotalCharges values:")
print(data.loc[problematic_indices, "TotalCharges"].unique())

#Full rows to find patterns
print("\nFull rows with non-numeric TotalCharges:")
print(data.loc[problematic_indices, ["customerID", "tenure", "MonthlyCharges", "TotalCharges"]])

Non-numeric TotalCharges values:
[' ']

Full rows with non-numeric TotalCharges:
      customerID  tenure  MonthlyCharges TotalCharges
488   4472-LVYGI       0           52.55             
753   3115-CZMZD       0           20.25             
936   5709-LVOEQ       0           80.85             
1082  4367-NUYAO       0           25.75             
1340  1371-DWPAZ       0           56.05             
3331  7644-OMVMY       0           19.85             
3826  3213-VVOLG       0           25.35             
4380  2520-SGTTA       0           20.00             
5218  2923-ARZLG       0           19.70             
6670  4075-WKNIU       0           73.35             
6754  2775-SEFEE       0           61.90             


In [38]:
 # Find all rows with tenure = 0
zero_tenure = data[data['tenure'] == 0]
print(f"Total rows with tenure = 0: {len(zero_tenure)}")

# Look at their TotalCharges values
print("\nTotalCharges values for tenure = 0 customers:")
print(zero_tenure['TotalCharges'].unique())

# See a few examples
print("\nSample of tenure = 0 customers:")
print(zero_tenure[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']].head())

# Test the reverse - do all empty TotalCharges have tenure = 0?
empty_total_charges = data.loc[problematic_indices]
print(f"\nTenure values for rows with non-numeric TotalCharges:")
print(empty_total_charges['tenure'].unique())

Total rows with tenure = 0: 11

TotalCharges values for tenure = 0 customers:
[' ']

Sample of tenure = 0 customers:
      customerID  tenure  MonthlyCharges TotalCharges
488   4472-LVYGI       0           52.55             
753   3115-CZMZD       0           20.25             
936   5709-LVOEQ       0           80.85             
1082  4367-NUYAO       0           25.75             
1340  1371-DWPAZ       0           56.05             

Tenure values for rows with non-numeric TotalCharges:
[0]


In [39]:
 # Handle missing TotalCharges values
# Analysis showed 11 customers with tenure=0 have empty string TotalCharges
# Business logic: New customers (0 months) haven't accumulated charges yet

# Decision: Set TotalCharges = 0 for these customers
# Rationale: Most conservative approach, assumes TotalCharges represents
# cumulative billing over complete months only
# Alternative considered: Set to MonthlyCharges (assumes partial month billing)
# In production: Would validate billing logic with business stakeholders

print("Before cleaning:")
print(f"Non-numeric TotalCharges: {pd.to_numeric(data['TotalCharges'], errors='coerce').isna().sum()}")

# Apply the fix
data.loc[data['tenure'] == 0, 'TotalCharges'] = '0'
data['TotalCharges'] = pd.to_numeric(data['TotalCharges'])

print("After cleaning:")
print(f"TotalCharges data type: {data['TotalCharges'].dtype}")
print(f"Missing values: {data['TotalCharges'].isna().sum()}")
print(f"TotalCharges for tenure=0 customers: {data.loc[data['tenure']==0, 'TotalCharges'].unique()}")

Before cleaning:
Non-numeric TotalCharges: 11
After cleaning:
TotalCharges data type: float64
Missing values: 0
TotalCharges for tenure=0 customers: [0.]


In [42]:
#Does TotalCharges generally align with tenure * MonthlyCharges?
data['Expected_Total'] = data['tenure'] * data['MonthlyCharges']
print("Sample of TotalCharges vs Expected (tenure*monthly):")
print(data[['tenure', 'MonthlyCharges', 'TotalCharges', 'Expected_Total']].head(10))

Sample of TotalCharges vs Expected (tenure*monthly):
   tenure  MonthlyCharges  TotalCharges  Expected_Total
0       1           29.85         29.85           29.85
1      34           56.95       1889.50         1936.30
2       2           53.85        108.15          107.70
3      45           42.30       1840.75         1903.50
4       2           70.70        151.65          141.40
5       8           99.65        820.50          797.20
6      22           89.10       1949.40         1960.20
7      10           29.75        301.90          297.50
8      28          104.80       3046.05         2934.40
9      62           56.15       3487.95         3481.30
